# BRENDA — source-context and interpretation audit

This supplementary analysis reviews the raw fields behind the common contract.
It does not modify normalized data, identify cleavage positions or look up
substrates in MEROPS. The adapter's concrete decisions are recorded by JSON
locator in `normalized/remarks_brenda.md`.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

here = Path.cwd().resolve()
ROOT = next(p for p in [here, *here.parents] if (p / "assets/tools/proteolysis_data_manager").is_dir())
sys.path.insert(0, str(ROOT / "assets/tools/proteolysis_data_manager/src"))
from proteolysis_data_manager.integration.kinetics.reader import load_kinetics_data
from proteolysis_data_manager.analysis.kinetics.overview import (
    table_inventory, missingness, species_coverage, parameter_summary,
    formatted_table, plot_parameters, plot_conditions,
)
DATA_ROOT = ROOT / "data/kinetics_datasets"
SERIES = ROOT / "assets/experiments/analysis/09_26/20_serum_proteolysis_calibration_max/notebooks/0_data_analysis_kinetics"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": .12})
def table(frame, limit=30):
    if len(frame) > limit:
        print(f"Preview: {limit} of {len(frame)} rows")
    display(HTML(formatted_table(frame, limit)))
def figure(fig):
    plt.show()
    plt.close(fig)

In [2]:
dataset = load_kinetics_data(DATA_ROOT)
view = dataset.measurements_view()
table(view.loc[view.value_relation.ne("eq"), ["measurement_id", "parameter_type", "value_relation", "value_numeric", "source_record_id"]], 25)
table(pd.DataFrame([{"scope": "All parameter families", "non_point_or_unresolved": int(view.value_numeric.isna().sum()),
    "reported_CI": int(view.ci_lower.notna().sum()), "reported_SD": int(view.sd.notna().sum())}]))

Preview: 25 of 792 rows


measurement_id,parameter_type,value_relation,value_numeric,source_record_id
brenda:measurement:79f3b4848452f002501aa7c0,kcat,range,None,"[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/66'}]"
brenda:measurement:ff3d247907dbc418f483f35f,kcat,unknown,None,"[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/69'}]"
brenda:measurement:cf892dcc97db238b8a09f254,kcat,range,None,"[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/70'}]"
brenda:measurement:c643a469140c5fbbad59f11e,km,range,None,"[{'source': 'brenda', 'id': 'data/3.4.11.1/km_value/54'}]"
brenda:measurement:f1d168b67ef91601e6f615a0,km,unknown,None,"[{'source': 'brenda', 'id': 'data/3.4.11.1/km_value/132'}]"
brenda:measurement:16e11da5ef26b5e1dfe65c9b,km,unknown,None,"[{'source': 'brenda', 'id': 'data/3.4.11.1/km_value/133'}]"
brenda:measurement:abed6ae56d806d52b204b5ea,km,unknown,None,"[{'source': 'brenda', 'id': 'data/3.4.11.1/km_value/134'}]"
brenda:measurement:3a16f09e38ff41b1e92677d3,km,unknown,None,"[{'source': 'brenda', 'id': 'data/3.4.11.1/km_value/135'}]"
brenda:measurement:34716849cea389b1045bf040,kcat,unknown,None,"[{'source': 'brenda', 'id': 'data/3.4.11.10/turnover_number/11'}]"
brenda:measurement:163141fce8fcf420b8a212ee,kcat,range,None,"[{'source': 'brenda', 'id': 'data/3.4.11.10/turnover_number/55'}]"


scope,non_point_or_unresolved,reported_CI,reported_SD
All parameter families,792,0,0


## Read source entries without cross-record pairing

Each locator identifies the source entry, not an independent experiment. The
same protein and PMID can describe several mutants, substrates, pH values or
temperatures. Protein-group associations and optimum fields must not silently
populate a specific assay. The rows below preserve the original value/comment. Unusual temperatures such
as the source entry reporting 378°C are retained for review, not corrected.

In [3]:
from proteolysis_data_manager.acquisition.kinetics.brenda import verify_snapshot
from proteolysis_data_manager.integration.kinetics.adapters.brenda import iter_records, PARAMETERS
receipt = verify_snapshot(DATA_ROOT)
raw_rows = []
for ec, record in iter_records(DATA_ROOT / receipt["raw_path"]):
    if not ec.startswith("3.4."):
        continue
    for family in PARAMETERS:
        for index, entry in enumerate(record.get(family) or []):
            raw_rows.append({"source_locator": f"data/{ec}/{family}/{index}", "family": family,
                "value_raw": str(entry.get("value") or ""), "comment_raw": str(entry.get("comment") or ""),
                "protein_links": len(set(map(str, entry.get("proteins") or []))),
                "reference_links": len(set(map(str, entry.get("references") or [])))})
raw = pd.DataFrame(raw_rows)
assert len(raw) == len(view)
table(raw.loc[raw.protein_links.ne(1)], 15)
table(raw.loc[raw.value_raw.str.contains(r"^-|[<>±]|\d\s*[-–]\s*\d", regex=True)], 15)

Preview: 15 of 199 rows


source_locator,family,value_raw,comment_raw,protein_links,reference_links
data/3.4.11.1/turnover_number/65,turnover_number,2.94 {L-Ala-7-amido-4-methylcoumarin},"pH 8.0, 37°C, recombinant His-tagged enzyme",3,1
data/3.4.11.1/turnover_number/66,turnover_number,3-6 {L-Cys-7-amido-4-methylcoumarin},"pH 8.0, 37°C, recombinant His-tagged enzyme",2,1
data/3.4.11.1/turnover_number/67,turnover_number,3.3 {L-Met-7-amido-4-methylcoumarin},"pH 8.0, 37°C, recombinant His-tagged enzyme",2,1
data/3.4.11.1/turnover_number/76,turnover_number,0.19 {L-Ile-7-amido-4-methylcoumarin},"pH 8.0, 37°C, recombinant His-tagged enzyme",2,1
data/3.4.11.1/turnover_number/152,turnover_number,0.12 {L-Ala-4-nitroanilide},"pH 8, 20°C",2,1
data/3.4.11.1/turnover_number/153,turnover_number,0.02 {L-Pro-4-nitroanilide},"pH 8, 20°C",2,1
data/3.4.11.1/km_value/63,km_value,7.2 {L-Leu-Gly},similar value,2,2
data/3.4.11.1/km_value/101,km_value,4.02 {leucinamide},similar value,2,2
data/3.4.11.1/km_value/132,km_value,-999 {more},kinetics,2,2
data/3.4.11.10/turnover_number/11,turnover_number,-999 {more},penultimative substituents influence rate of hydrolysis,2,2


Preview: 15 of 1032 rows


source_locator,family,value_raw,comment_raw,protein_links,reference_links
data/3.4.11.1/turnover_number/66,turnover_number,3-6 {L-Cys-7-amido-4-methylcoumarin},"pH 8.0, 37°C, recombinant His-tagged enzyme",2,1
data/3.4.11.1/turnover_number/69,turnover_number,-999 {L-Met-7-amido-4-methylcoumarin},"pH 8.0, 37°C, recombinant His-tagged enzyme",1,1
data/3.4.11.1/turnover_number/70,turnover_number,0.057-0.65 {L-Leu-7-amido-4-methylcoumarin},"pH 8.0, 37°C, recombinant His-tagged enzyme",1,1
data/3.4.11.1/km_value/54,km_value,0.2-1 {beta-cyclohexyl-L-alanyl-4-methylcoumaryl-7-amide},"pH 8.0, 37°C",1,1
data/3.4.11.1/km_value/132,km_value,-999 {more},kinetics,2,2
data/3.4.11.1/km_value/133,km_value,-999 {more},Michaelis-Menten kinetics,1,1
data/3.4.11.1/km_value/134,km_value,-999 {more},"metal ion binding kinetics to the recombinant enzyme, metal ion occupying site 1 can influence enzyme substrate kinetics, overview",1,1
data/3.4.11.1/km_value/135,km_value,-999 {more},"dynamics of ACTH-(6-9)-PGP peptide hydrolysis with leucine aminopeptidase and carboxypeptidase Y and reaction kinetics, overview",1,1
data/3.4.11.10/turnover_number/11,turnover_number,-999 {more},penultimative substituents influence rate of hydrolysis,2,2
data/3.4.11.10/turnover_number/55,turnover_number,50-69 {L-leucine 4-nitroanilide},"native enzyme, buffer dependent",1,1


## Reported mentions are not automatically resolved annotations

These lexical counts are inspection aids, not extracted cleavage sites or
assay classifications. In particular, a comment containing “cleavage” does not
establish a position or link a global kinetic parameter to that position.

In [4]:
patterns = {"cleavage_or_hydrolysis_mention": r"cleav|scissile|hydrolys",
    "serum_or_plasma_mention": r"serum|plasma", "buffer_mention": r"buffer",
    "variant_mention": r"mutant|wild.type|recombinant|variant",
    "optimum_or_storage_mention": r"optim|stability|storage|stored"}
table(pd.DataFrame([{"description": name, "source_entries": int(raw.comment_raw.str.contains(pattern, case=False, regex=True).sum())}
                    for name, pattern in patterns.items()]))
site_mentions = raw.loc[raw.comment_raw.str.contains(patterns["cleavage_or_hydrolysis_mention"], case=False, regex=True)]
table(site_mentions, 20)
table(raw.loc[raw.comment_raw.str.contains(patterns["serum_or_plasma_mention"], case=False, regex=True)], 20)

description,source_entries
cleavage_or_hydrolysis_mention,521
serum_or_plasma_mention,104
buffer_mention,883
variant_mention,8360
optimum_or_storage_mention,14


Preview: 20 of 521 rows


source_locator,family,value_raw,comment_raw,protein_links,reference_links
data/3.4.11.1/km_value/135,km_value,-999 {more},"dynamics of ACTH-(6-9)-PGP peptide hydrolysis with leucine aminopeptidase and carboxypeptidase Y and reaction kinetics, overview",1,1
data/3.4.11.10/turnover_number/11,turnover_number,-999 {more},penultimative substituents influence rate of hydrolysis,2,2
data/3.4.11.2/km_value/147,km_value,-999 {amino acid amides and amino acyl-p-nitroanilides},influence of organic solvents on substrate binding and hydrolysis,1,1
data/3.4.11.5/km_value/21,km_value,-999 {more},"no hydrolysis of Glu-p-nitroanilide, Gly-p-nitroanilide, Phe-p-nitroanilide",1,1
data/3.4.19.9/turnover_number/11,turnover_number,-999 {more},pteroyltriglutamate: 732 gamma-glutamyl bonds are cleaved per min per mol of enzyme,1,1
data/3.4.21.1/km_value/33,km_value,-999 {casein},"hydrolysis kinetics are analysed in the framework of a two-step model, with consecutive demasking and hydrolysis steps. Determination of kinetic parameters for casein hydrolysis by chymotrypsin using two ranges of substrate concentration",1,1
data/3.4.21.109/turnover_number/32,turnover_number,59 {methylsulfonyl-D-cyclohexyltyrosyl-glycyl-arginine-p-nitroanilide},pseudozymogen His6t-S-CD consisting of a spacer and the catalytical domain with an N-terminal His6-tag and a cleavage site for activation by enterokinase is used,1,1
data/3.4.21.109/km_value/42,km_value,0.17 {methylsulfonyl-D-cyclohexyltyrosyl-glycyl-arginine-p-nitroanilide},pseudozymogen His6t-S-CD consisting of a spacer and the catalytical domain with an N-terminal His6-tag and a cleavage site for activation by enterokinase is used,1,1
data/3.4.21.109/kcat_km_value/3,kcat_km_value,350 {methylsulfonyl-D-cyclohexyltyrosyl-glycyl-arginine-p-nitroanilide},pseudozymogen His6t-S-CD consisting of a spacer and the catalytical domain with an N-terminal His6-tag and a cleavage site for activation by enterokinase is used,1,1
data/3.4.21.4/turnover_number/125,turnover_number,223 {Suc-Ala-Ala-Pro-Arg-4-nitroanilide},"pH 8.0, 37°C, uncleaved enzyme, nonsulfated",1,1


Preview: 20 of 104 rows


source_locator,family,value_raw,comment_raw,protein_links,reference_links
data/3.4.11.20/km_value/1,km_value,0.01 {Leu 4-methylcoumarin 7-amide},plasma enzyme,1,1
data/3.4.11.7/km_value/78,km_value,0.72-0.846 {Asp-beta-naphthylamide},maternal and normal serum,1,1
data/3.4.15.1/km_value/29,km_value,2.9 {hippuryl-His-Leu},serum enzyme,1,1
data/3.4.15.1/km_value/31,km_value,0.076 {angiotensin I},serum enzyme,1,1
data/3.4.17.20/turnover_number/8,turnover_number,40.7 {Hippuryl-L-Lys},plasma-derived enzyme,1,1
data/3.4.17.20/turnover_number/14,turnover_number,52.4 {hippuryl-L-Arg},plasma-derived enzyme,1,1
data/3.4.17.20/turnover_number/15,turnover_number,293 {N-[3-(2-Furylacryloyl)]-L-Ala-L-Lys},plasma-derived enzyme,1,1
data/3.4.17.20/km_value/20,km_value,4.71 {N-[3-(2-Furylacryloyl)]-L-Ala-L-Lys},plasma-derived enzyme,1,1
data/3.4.17.20/km_value/24,km_value,766 {Hippuryl-L-Lys},plasma-derived enzyme,1,1
data/3.4.17.20/km_value/26,km_value,817 {hippuryl-L-Arg},plasma-derived enzyme,1,1


## Unit and condition rules

`Km`: mM → M (×0.001). `kcat`: s⁻¹. `kcat/Km`: inverse mM/time →
M⁻¹s⁻¹ (×1000). These are family defaults, not ratios computed from unrelated
rows. BRENDA has an inconsistent `mM/s` heading on its field-definition page;
the [search form](https://www.brenda-enzymes.org/search_result.php?a=305) specifies
inverse concentration/time. The manifest records the exact conversion rule.

Only a single unambiguous pH/°C scalar in the entry comment is read. Ranges,
conflicting mentions and optimum/storage/stability context remain null. General
enzyme optima and source tissue are not assay metadata.

**TODO:** review assay-method, matrix/mode, concentration and reported uncertainty
entry by entry where the source supplies evidence. This is curation, not imputation.
Do not introduce fragment searches or cross-family chemistry enrichment.

In [5]:
table(pd.DataFrame(dataset.metadata["unit_policy"].items(), columns=["source_field", "conversion"]))
table(view.loc[view.ph.notna() | view.temperature_c.notna(), ["parameter_type", "ph", "temperature_c", "conditions_raw", "source_record_id"]], 20)
table(view.loc[view.conditions_raw.fillna("").str.contains(r"optim|stability|storage|stored", case=False, regex=True),
               ["ph", "temperature_c", "conditions_raw", "source_record_id"]], 15)

source_field,conversion
turnover_number,s^-1 unchanged
km_value,mM -> M: multiply by 0.001
kcat_km_value,(mM*s)^-1 -> (M*s)^-1: multiply by 1000


Preview: 20 of 17374 rows


parameter_type,ph,temperature_c,conditions_raw,source_record_id
kcat,7.5,80,"pH 7.5, 80°C","[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/1'}]"
kcat,7.5,80,"pH 7.5, 80°C","[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/2'}]"
kcat,7.5,80,"pH 7.5, 80°C","[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/3'}]"
kcat,7.5,80,"pH 7.5, 80°C","[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/4'}]"
kcat,7.5,35,"pH 7.5, 35°C","[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/5'}]"
kcat,7.5,35,"pH 7.5, 35°C","[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/6'}]"
kcat,8,37,"pH 8.0, 37°C, recombinant enzyme","[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/7'}]"
kcat,8,37,"pH 8.0, 37°C, recombinant enzyme","[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/8'}]"
kcat,8,37,"pH 8.0, 37°C, recombinant enzyme","[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/9'}]"
kcat,8,37,"pH 8.0, 37°C","[{'source': 'brenda', 'id': 'data/3.4.11.1/turnover_number/10'}]"


ph,temperature_c,conditions_raw,source_record_id
None,None,kinetics and thermodynamic stability,"[{'source': 'brenda', 'id': 'data/3.4.21.19/km_value/10'}]"
None,None,"although neither plasmin nor urokinase-type plasminogen activator exhibits allosteric cooperativity, modeling shows that cooperativity occurs at the system level because of substrate competition, molecular modeling and simulation, substrate competition and bistability, overview","[{'source': 'brenda', 'id': 'data/3.4.21.7/km_value/37'}]"
None,None,"overview: turnover numbers for the hydrolysis of the peptides of the series Ac-(Ala)m-Lys-(4-nitro)Phe-(Ala)n amide and of the series Ac-Xaa-Lys-(4-nitro)Phe-Ala-Ala amide at the pH optimum for kcat, at pH 5.5 and pH 6.0","[{'source': 'brenda', 'id': 'data/3.4.23.20/turnover_number/0'}]"
None,None,pH 5.5 (pH-optimum for kcat),"[{'source': 'brenda', 'id': 'data/3.4.23.20/turnover_number/24'}]"
None,None,pH 4.5 (pH-optimum for kcat),"[{'source': 'brenda', 'id': 'data/3.4.23.20/turnover_number/33'}]"
None,None,pH 4.3 (pH-optimum for kcat),"[{'source': 'brenda', 'id': 'data/3.4.23.20/turnover_number/40'}]"
None,None,pH 3.8-4.2 (pH-optimum for kcat),"[{'source': 'brenda', 'id': 'data/3.4.23.20/turnover_number/41'}]"
None,None,"overview: Km values for the hydrolysis of the peptides of the series Ac-(Ala)m-Lys-(4-nitro)Phe-(Ala)n amide and of the series Ac-Xaa-Lys-(4-nitro)Phe-Ala-Ala amide at the pH optimum for kcat, at pH 5.5 and pH 6.0","[{'source': 'brenda', 'id': 'data/3.4.23.20/km_value/0'}]"
None,None,pH 4.3 (optimum for kcat),"[{'source': 'brenda', 'id': 'data/3.4.23.20/km_value/5'}]"
None,None,pH 3.8-4.2 (optimum for kcat),"[{'source': 'brenda', 'id': 'data/3.4.23.20/km_value/13'}]"
